# Notebook 01: Ingest XuetangX

**Purpose:** Parse raw XuetangX JSON file → normalized Parquet → DuckDB view.

**Inputs:**
- `data/raw/xuetangx/20170201-20170801-raw_user_activity.json` (~5GB, Feb-Aug 2017)

**Outputs:**
- `data/interim/xuetangx_events_raw.parquet` (28M learning events)
- `data/interim/xuetangx.duckdb` (view: `xuetangx_events_raw`)
- `reports/01_ingest_xuetangx/<run_tag>/report.json`

**Strategy:**
- Keep platform sessions (session_hash)
- Filter learning events only (load_*, problem_*, video_*, seek_*, speed_*, pause_*)
- JSON → Parquet → DuckDB (reproducible, Windows-safe)

In [1]:
# [CELL 01-00] Bootstrap: repo root + paths + logger

import os
import sys
import json
import time
import uuid
import hashlib
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List

import numpy as np
import pandas as pd

t0 = datetime.now()
print(f"[CELL 01-00] start={t0.isoformat(timespec='seconds')}")
print("[CELL 01-00] CWD:", Path.cwd().resolve())

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "PROJECT_STATE.md").exists():
            return p
    raise RuntimeError("Could not find PROJECT_STATE.md. Open notebook from within the repo.")

REPO_ROOT = find_repo_root(Path.cwd())
print("[CELL 01-00] REPO_ROOT:", REPO_ROOT)

PATHS = {
    "PROJECT_STATE": REPO_ROOT / "PROJECT_STATE.md",
    "META_REGISTRY": REPO_ROOT / "meta.json",
    "DATA_RAW": REPO_ROOT / "data" / "raw",
    "DATA_INTERIM": REPO_ROOT / "data" / "interim",
    "DATA_PROCESSED": REPO_ROOT / "data" / "processed",
    "REPORTS": REPO_ROOT / "reports",
}
for k, v in PATHS.items():
    print(f"[CELL 01-00] {k}={v}")

def cell_start(cell_id: str, title: str, **kwargs: Any) -> float:
    t = time.time()
    print(f"\n[{cell_id}] {title}")
    print(f"[{cell_id}] start={datetime.now().isoformat(timespec='seconds')}")
    for k, v in kwargs.items():
        print(f"[{cell_id}] {k}={v}")
    return t

def cell_end(cell_id: str, t0: float, **kwargs: Any) -> None:
    for k, v in kwargs.items():
        print(f"[{cell_id}] {k}={v}")
    print(f"[{cell_id}] elapsed={time.time()-t0:.2f}s")
    print(f"[{cell_id}] done")

print("[CELL 01-00] done")

[CELL 01-00] start=2026-04-08T12:52:23
[CELL 01-00] CWD: /home/user/jamalla/anonymous-users-mooc-session-meta/notebooks/xuetangx
[CELL 01-00] REPO_ROOT: /home/user/jamalla/anonymous-users-mooc-session-meta
[CELL 01-00] PROJECT_STATE=/home/user/jamalla/anonymous-users-mooc-session-meta/PROJECT_STATE.md
[CELL 01-00] META_REGISTRY=/home/user/jamalla/anonymous-users-mooc-session-meta/meta.json
[CELL 01-00] DATA_RAW=/home/user/jamalla/anonymous-users-mooc-session-meta/data/raw
[CELL 01-00] DATA_INTERIM=/home/user/jamalla/anonymous-users-mooc-session-meta/data/interim
[CELL 01-00] DATA_PROCESSED=/home/user/jamalla/anonymous-users-mooc-session-meta/data/processed
[CELL 01-00] REPORTS=/home/user/jamalla/anonymous-users-mooc-session-meta/reports
[CELL 01-00] done


In [2]:
# [CELL 01-01] Reproducibility: seed everything

t0 = cell_start("CELL 01-01", "Seed everything")

GLOBAL_SEED = 20260107

def seed_everything(seed: int) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)

seed_everything(GLOBAL_SEED)

cell_end("CELL 01-01", t0, seed=GLOBAL_SEED)


[CELL 01-01] Seed everything
[CELL 01-01] start=2026-04-08T12:52:23
[CELL 01-01] seed=20260107
[CELL 01-01] elapsed=0.00s
[CELL 01-01] done


In [3]:
# [CELL 01-02] JSON IO + hashing helpers

t0 = cell_start("CELL 01-02", "JSON IO + hashing")

def write_json_atomic(path: Path, obj: Any, indent: int = 2) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp_{uuid.uuid4().hex}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=indent)
    tmp.replace(path)

def read_json(path: Path) -> Any:
    if not path.exists():
        raise RuntimeError(f"Missing JSON file: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def assert_nonempty_df(df: pd.DataFrame, name: str) -> None:
    if df is None or not isinstance(df, pd.DataFrame) or df.shape[0] == 0:
        raise RuntimeError(f"{name} is empty or invalid DataFrame")

cell_end("CELL 01-02", t0)


[CELL 01-02] JSON IO + hashing
[CELL 01-02] start=2026-04-08T12:52:23
[CELL 01-02] elapsed=0.00s
[CELL 01-02] done


In [4]:
# [CELL 01-03] Run tagging + report/config/manifest + meta.json

t0 = cell_start("CELL 01-03", "Start run + init run files + meta.json")

NOTEBOOK_NAME = "01_ingest_xuetangx"
DATASET = "xuetangx"
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID = uuid.uuid4().hex

OUT_DIR = PATHS["REPORTS"] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH   = OUT_DIR / "report.json"
CONFIG_PATH   = OUT_DIR / "config.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

RAW_DIR     = PATHS["DATA_RAW"] / "xuetangx"
OUT_PARQUET = PATHS["DATA_INTERIM"] / "xuetangx_events_raw.parquet"
OUT_DUCKDB  = PATHS["DATA_INTERIM"] / "xuetangx.duckdb"

CFG = {
    "notebook": NOTEBOOK_NAME,
    "dataset": DATASET,
    "run_id": RUN_ID,
    "run_tag": RUN_TAG,
    "seed": GLOBAL_SEED,
    "paths": {
        "raw_dir": str(RAW_DIR),
        "out_parquet": str(OUT_PARQUET),
        "out_duckdb": str(OUT_DUCKDB),
        "out_dir": str(OUT_DIR),
    },
    "ingest": {
        "source_file": "20170201-20170801-raw_user_activity.json",
        "learning_events_only": True,
        "learning_event_prefixes": ["load_", "problem_", "video_", "seek_", "speed_", "pause_"],
        "parquet_compression": "zstd",
        "duckdb_view": "xuetangx_events_raw",
    }
}

write_json_atomic(CONFIG_PATH, CFG)

report = {
    "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "repo_root": str(REPO_ROOT), "metrics": {}, "key_findings": [],
    "sanity_samples": {}, "data_fingerprints": {}, "notes": [],
}
write_json_atomic(REPORT_PATH, report)

manifest = {"run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG, "artifacts": []}
write_json_atomic(MANIFEST_PATH, manifest)

META_PATH = PATHS["META_REGISTRY"]
if not META_PATH.exists():
    write_json_atomic(META_PATH, {"schema_version": 1, "runs": []})
meta = read_json(META_PATH)
meta["runs"].append({"run_id": RUN_ID, "notebook": NOTEBOOK_NAME,
                      "run_tag": RUN_TAG, "out_dir": str(OUT_DIR),
                      "created_at": datetime.now().isoformat(timespec="seconds")})
write_json_atomic(META_PATH, meta)

cell_end("CELL 01-03", t0, out_dir=str(OUT_DIR))


[CELL 01-03] Start run + init run files + meta.json
[CELL 01-03] start=2026-04-08T12:52:23
[CELL 01-03] out_dir=/home/user/jamalla/anonymous-users-mooc-session-meta/reports/01_ingest_xuetangx/20260408_125223
[CELL 01-03] elapsed=0.01s
[CELL 01-03] done


In [5]:
# [CELL 01-04] Enumerate raw JSON files

t0 = cell_start("CELL 01-04", "Enumerate raw JSON files", raw_dir=str(RAW_DIR))

if not RAW_DIR.exists():
    raise RuntimeError(f"Raw directory not found: {RAW_DIR}")

source_file = RAW_DIR / CFG["ingest"]["source_file"]
if not source_file.exists():
    raise RuntimeError(
        f"Source file not found: {source_file}\n"
        f"Expected: data/raw/xuetangx/20170201-20170801-raw_user_activity.json"
    )

print(f"[CELL 01-04] Found: {source_file.name} ({source_file.stat().st_size / 1024**3:.2f} GB)")

cell_end("CELL 01-04", t0)


[CELL 01-04] Enumerate raw JSON files
[CELL 01-04] start=2026-04-08T12:52:23
[CELL 01-04] raw_dir=/home/user/jamalla/anonymous-users-mooc-session-meta/data/raw/xuetangx
[CELL 01-04] Found: 20170201-20170801-raw_user_activity.json (4.79 GB)
[CELL 01-04] elapsed=0.00s
[CELL 01-04] done


In [6]:
# [CELL 01-05] Parse nested JSON → flattened DataFrame

t0 = cell_start("CELL 01-05", "Parse nested JSON to DataFrame")

LEARNING_PREFIXES = tuple(CFG["ingest"]["learning_event_prefixes"])

def parse_xuetangx_json(file_path: Path) -> pd.DataFrame:
    """
    Parse XuetangX JSON structure:
    [[course_id, {user_id: {session_hash: [[event, timestamp], ...]}}], ...]
    Returns: DataFrame with [course_id, user_id, session_hash, event_type, timestamp]
    """
    print(f"  Parsing: {file_path.name} ({file_path.stat().st_size / 1024**2:.1f} MB)...")

    with file_path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    rows = []
    n_courses = len(data)
    n_total = 0
    n_kept = 0

    for idx, course_data in enumerate(data):
        if (idx + 1) % 200 == 0:
            print(f"    Progress: {idx + 1}/{n_courses} courses...")
        course_id = course_data[0]
        for user_id, sessions_dict in course_data[1].items():
            for session_hash, events_list in sessions_dict.items():
                for event_data in events_list:
                    event_type, timestamp = event_data[0], event_data[1]
                    n_total += 1
                    if event_type.startswith(LEARNING_PREFIXES):
                        rows.append({
                            "course_id": course_id,
                            "user_id": user_id,
                            "session_hash": session_hash,
                            "event_type": event_type,
                            "timestamp": timestamp,
                        })
                        n_kept += 1

    print(f"    Events: {n_total:,} total → {n_kept:,} kept (learning only)")
    return pd.DataFrame(rows)

events = parse_xuetangx_json(source_file)
assert_nonempty_df(events, "events")

print(f"\n[CELL 01-05] Shape: {events.shape}")
print(f"[CELL 01-05] Columns: {list(events.columns)}")
print(events.head(3).to_string(index=False))

cell_end("CELL 01-05", t0, rows=int(events.shape[0]))


[CELL 01-05] Parse nested JSON to DataFrame
[CELL 01-05] start=2026-04-08T12:52:23
  Parsing: 20170201-20170801-raw_user_activity.json (4903.7 MB)...
    Progress: 200/1577 courses...
    Progress: 400/1577 courses...
    Progress: 600/1577 courses...
    Progress: 800/1577 courses...
    Progress: 1000/1577 courses...
    Progress: 1200/1577 courses...
    Progress: 1400/1577 courses...
    Events: 56,985,474 total → 28,002,537 kept (learning only)

[CELL 01-05] Shape: (28002537, 5)
[CELL 01-05] Columns: ['course_id', 'user_id', 'session_hash', 'event_type', 'timestamp']
                              course_id user_id                     session_hash  event_type           timestamp
course-v1:TsinghuaX+10610204_tv+2015_T1 1660673 22ef2c411f8cf49bfeeca748ce280679  load_video 2017-05-22T18:23:06
course-v1:TsinghuaX+10610204_tv+2015_T1 1660673 22ef2c411f8cf49bfeeca748ce280679 pause_video 2017-05-22T18:23:08
course-v1:TsinghuaX+10610204_tv+2015_T1 2021250 125ca511da0b5b5c8bd9b84294d22763 

In [7]:
# [CELL 01-06] Save canonical Parquet

t0 = cell_start("CELL 01-06", "Save canonical Parquet", out_parquet=str(OUT_PARQUET))

OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
events.to_parquet(OUT_PARQUET, index=False, compression=CFG["ingest"]["parquet_compression"])

parq_bytes = int(OUT_PARQUET.stat().st_size)
parq_sha   = sha256_file(OUT_PARQUET)

print(f"[CELL 01-06] Saved: {OUT_PARQUET}")
print(f"[CELL 01-06] Size:  {parq_bytes / 1024**2:.1f} MB")
print(f"[CELL 01-06] SHA256: {parq_sha}")

cell_end("CELL 01-06", t0)


[CELL 01-06] Save canonical Parquet
[CELL 01-06] start=2026-04-08T12:53:57
[CELL 01-06] out_parquet=/home/user/jamalla/anonymous-users-mooc-session-meta/data/interim/xuetangx_events_raw.parquet
[CELL 01-06] Saved: /home/user/jamalla/anonymous-users-mooc-session-meta/data/interim/xuetangx_events_raw.parquet
[CELL 01-06] Size:  94.7 MB
[CELL 01-06] SHA256: 14891bb4405808d0db3d23c15654c1b88712c6f0c86e6d50274c13c9ccc2f824
[CELL 01-06] elapsed=15.59s
[CELL 01-06] done


In [8]:
# [CELL 01-07] DuckDB: create DB + view from Parquet

t0 = cell_start("CELL 01-07", "Create DuckDB + view", out_duckdb=str(OUT_DUCKDB))

try:
    import duckdb
except ImportError as e:
    raise RuntimeError("Missing duckdb. Install via: pip install duckdb") from e

OUT_DUCKDB.parent.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(OUT_DUCKDB))

view = CFG["ingest"]["duckdb_view"]
con.execute(f"DROP VIEW IF EXISTS {view};")
con.execute(f"""
    CREATE VIEW {view} AS
    SELECT * FROM read_parquet('{str(OUT_PARQUET).replace("'", "''")}')""")

n = con.execute(f"SELECT COUNT(*) FROM {view}").fetchone()[0]
schema_df = con.execute(f"DESCRIBE {view}").fetchdf()
con.close()

print(f"[CELL 01-07] View: {view}")
print(f"[CELL 01-07] Rows: {int(n):,}")
print(schema_df.to_string(index=False))

cell_end("CELL 01-07", t0, rows=int(n))


[CELL 01-07] Create DuckDB + view
[CELL 01-07] start=2026-04-08T12:54:13
[CELL 01-07] out_duckdb=/home/user/jamalla/anonymous-users-mooc-session-meta/data/interim/xuetangx.duckdb
[CELL 01-07] View: xuetangx_events_raw
[CELL 01-07] Rows: 28,002,537
 column_name column_type null  key default extra
   course_id     VARCHAR  YES None    None  None
     user_id     VARCHAR  YES None    None  None
session_hash     VARCHAR  YES None    None  None
  event_type     VARCHAR  YES None    None  None
   timestamp     VARCHAR  YES None    None  None
[CELL 01-07] rows=28002537
[CELL 01-07] elapsed=1.38s
[CELL 01-07] done


In [9]:
# [CELL 01-08] Basic stats (users / courses / sessions / events)

t0 = cell_start("CELL 01-08", "Compute basic stats")

stats = {
    "n_events":      int(events.shape[0]),
    "n_users":       int(events["user_id"].nunique()),
    "n_courses":     int(events["course_id"].nunique()),
    "n_sessions":    int(events["session_hash"].nunique()),
    "n_event_types": int(events["event_type"].nunique()),
}

print("[CELL 01-08] Stats:")
for k, v in stats.items():
    print(f"  {k}: {v:,}")

top_events = events["event_type"].value_counts().head(10)
print("\n[CELL 01-08] Top 10 event types:")
print(top_events.to_string())

# Expected: 28,002,537 events, 1,518 courses
assert stats["n_events"] == 28_002_537, f"Expected 28002537 events, got {stats['n_events']}"
assert stats["n_courses"] == 1_518, f"Expected 1518 courses, got {stats['n_courses']}"
print("\n[CELL 01-08] Assertions passed.")

cell_end("CELL 01-08", t0, **stats)


[CELL 01-08] Compute basic stats
[CELL 01-08] start=2026-04-08T12:54:14
[CELL 01-08] Stats:
  n_events: 28,002,537
  n_users: 182,755
  n_courses: 1,518
  n_sessions: 467,105
  n_event_types: 8

[CELL 01-08] Top 10 event types:
event_type
pause_video                10447067
seek_video                  4878540
problem_get                 4594062
load_video                  3832977
problem_check               1589086
problem_check_correct       1287667
problem_check_incorrect      717231
problem_save                 655907

[CELL 01-08] Assertions passed.
[CELL 01-08] n_events=28002537
[CELL 01-08] n_users=182755
[CELL 01-08] n_courses=1518
[CELL 01-08] n_sessions=467105
[CELL 01-08] n_event_types=8
[CELL 01-08] elapsed=10.03s
[CELL 01-08] done


In [10]:
# [CELL 01-09] Update report + manifest

t0 = cell_start("CELL 01-09", "Write fingerprints + sanity sample to report")

report = read_json(REPORT_PATH)
manifest = read_json(MANIFEST_PATH)

report["metrics"] = stats
report["sanity_samples"]["events_head3"] = events.head(3).to_dict(orient="records")
report["sanity_samples"]["top_event_types"] = events["event_type"].value_counts().head(10).to_dict()
report["data_fingerprints"]["source_file"] = {
    "name": source_file.name,
    "bytes": int(source_file.stat().st_size),
    "sha256": sha256_file(source_file),
}
report["data_fingerprints"]["xuetangx_events_raw_parquet"] = {
    "path": str(OUT_PARQUET), "bytes": parq_bytes, "sha256": parq_sha,
}
report["notes"].append(
    "Filtered to learning events only (load_*, problem_*, video_*, seek_*, speed_*, pause_*). "
    "Kept platform session_hash for sessionization in NB02."
)
write_json_atomic(REPORT_PATH, report)

def add_artifact(path: Path) -> None:
    try:
        sha = sha256_file(path)
    except Exception as e:
        sha = f"ERROR: {e}"
    manifest["artifacts"].append({"path": str(path), "bytes": int(path.stat().st_size), "sha256": sha})

add_artifact(OUT_PARQUET)
add_artifact(OUT_DUCKDB)
write_json_atomic(MANIFEST_PATH, manifest)

print(f"[CELL 01-09] Updated: {REPORT_PATH}")
print(f"[CELL 01-09] Updated: {MANIFEST_PATH}")

cell_end("CELL 01-09", t0)


[CELL 01-09] Write fingerprints + sanity sample to report
[CELL 01-09] start=2026-04-08T12:54:24
[CELL 01-09] Updated: /home/user/jamalla/anonymous-users-mooc-session-meta/reports/01_ingest_xuetangx/20260408_125223/report.json
[CELL 01-09] Updated: /home/user/jamalla/anonymous-users-mooc-session-meta/reports/01_ingest_xuetangx/20260408_125223/manifest.json
[CELL 01-09] elapsed=19.37s
[CELL 01-09] done


## Notebook 01 Complete

**Outputs:**
- `data/interim/xuetangx_events_raw.parquet` — 28,002,537 learning events
- `data/interim/xuetangx.duckdb` — view: `xuetangx_events_raw`
- `reports/01_ingest_xuetangx/<run_tag>/report.json`

**Next:** `02_sessionize_xuetangx.ipynb`
- Apply 30-min inactivity gap to form sessions
- Minimum 2 interactions per session